# 08. Resolución determinista y jerárquica de localizaciones

Este notebook transforma las menciones territoriales extraídas por IA en una jerarquía administrativa canónica basada en fuentes del Instituto Nacional de Estadística (INE):

1. **municipio**;
2. **provincia**;
3. **comunidad o ciudad autónoma**.

La resolución es determinista, trazable y conservadora. El contexto administrativo se calcula únicamente dentro de cada combinación `identificador_boe + event_id + project_mention_id`, se incorpora como metadato de auditoría en las filas resultantes y no se persiste como tabla independiente.

El estado global `location_resolution_status` mide la **completitud de la jerarquía**: solo es `resolved` cuando los tres niveles están resueltos.

## 1. Configuración e imports

Las operaciones transversales de nulos, normalización textual, validación de columnas y escritura Parquet se importan desde `renewables_permitting.utils`. Las funciones genéricas de comparación por tokens se importan desde `renewables_permitting.text_matching`. El notebook conserva únicamente la lógica específica de resolución administrativa e INE.


In [ ]:
import re
from dataclasses import dataclass
from difflib import SequenceMatcher
from enum import Enum
from pathlib import Path
from typing import Any

import pandas as pd
from pydantic import BaseModel, Field

from renewables_permitting.text_matching import (
    text_tokens,
    token_overlap_score,
)
from renewables_permitting.utils import (
    is_null_like,
    normalize_text_or_none,
    save_parquet,
    validate_required_columns,
)

# PROJECT_ROOT = Path(__file__).resolve().parents[2]  # fuera del notebook
PROJECT_ROOT = Path.cwd().parent  # dentro de notebooks/

DATA_DIR = PROJECT_ROOT / "data"
BRONZE_DIR = DATA_DIR / "bronze"
SILVER_DIR = DATA_DIR / "silver"

# Fuentes INE versionadas. Ajustar solo este directorio si ya están ubicadas
# en otra carpeta del proyecto.
BRONZE_INE_DIR = BRONZE_DIR / "localizaciones_ine"
INE_MUNICIPALITIES_SOURCE_PATH = BRONZE_INE_DIR / "diccionario26.csv"
INE_PROVINCES_SOURCE_PATH = (
    BRONZE_INE_DIR / "codine_ccaaprovincia_20260614.csv"
)

# Dimensión administrativa canónica.
DIM_MUNICIPALITIES_PATH = (
    SILVER_DIR / "dimensions" / "dim_municipalities.parquet"
)

# Localizaciones procedentes de la extracción con IA.
SILVER_BOE_AI_DIR = SILVER_DIR / "boe_ai"
PROJECT_LOCATIONS_PATH = SILVER_BOE_AI_DIR / "project_locations.parquet"

# Salidas del enriquecimiento determinista.
SILVER_DETERMINISTIC_ENRICHMENT_DIR = (
    SILVER_DIR / "boe_ai_deterministic_enrichment"
)
PROJECT_LOCATIONS_RESOLVED_PATH = (
    SILVER_DETERMINISTIC_ENRICHMENT_DIR
    / "project_locations_resolved.parquet"
)

LOCATION_RESOLVER_VERSION = "2026-07-10.5"
MUNICIPALITY_DIMENSION_SOURCE = (
    f"{INE_MUNICIPALITIES_SOURCE_PATH.name}|"
    f"{INE_PROVINCES_SOURCE_PATH.name}"
)



## 2. Construcción de la dimensión administrativa

La dimensión se regenera a partir de las dos fuentes INE versionadas. De este modo, los códigos se leen como cadenas, se preservan los ceros a la izquierda y se evita depender de una dimensión previa potencialmente obsoleta.


In [ ]:
def normalize_ine_code_series(values: pd.Series, *, width: int, column_name: str) -> pd.Series:
    normalized = values.astype('string').str.strip().str.replace(r'\.0$', '', regex=True)
    invalid = normalized.isna() | ~normalized.str.fullmatch(r'\d+')
    if invalid.any():
        examples = values.loc[invalid].head(5).tolist()
        raise ValueError(f'{column_name} contiene códigos no válidos: {examples}')
    if normalized.str.len().gt(width).any():
        examples = normalized.loc[normalized.str.len().gt(width)].head(5).tolist()
        raise ValueError(f'{column_name} excede la anchura {width}: {examples}')
    return normalized.str.zfill(width)


def normalized_name_variants(value: Any, *, aliases: set[str] | None = None) -> list[str]:
    if is_null_like(value):
        return []
    raw = str(value).strip()
    variants: set[str] = {raw}

    # Formas INE como "Pontes de García Rodríguez, As".
    parts = [part.strip() for part in raw.split(',') if part.strip()]
    if len(parts) == 2:
        variants.add(f'{parts[1]} {parts[0]}')

    # Denominaciones bilingües.
    for current in list(variants):
        slash_parts = [part.strip() for part in current.split('/') if part.strip()]
        variants.update(slash_parts)

    if aliases:
        variants.update(aliases)

    normalized = {normalize_text_or_none(item) for item in variants}
    return sorted(item for item in normalized if item is not None)


PROVINCE_ALIASES_BY_CODE = {
    '01': {'alava', 'araba'},
    '03': {'alicante', 'alacant'},
    '07': {'illes balears', 'islas baleares', 'baleares'},
    '12': {'castellon', 'castello'},
    '15': {'a coruna', 'la coruna', 'coruna'},
    '17': {'girona', 'gerona'},
    '20': {'gipuzkoa', 'guipuzcoa'},
    '26': {'la rioja', 'rioja'},
    '35': {'las palmas', 'palmas'},
    '46': {'valencia', 'valencia'},
    '48': {'bizkaia', 'vizcaya'},
}

AUTONOMOUS_COMMUNITY_ALIASES_BY_CODE = {
    '03': {'asturias', 'principado de asturias'},
    '04': {'illes balears', 'islas baleares', 'baleares'},
    '09': {'cataluna', 'catalunya'},
    '10': {'comunitat valenciana', 'comunidad valenciana'},
    '13': {'madrid', 'comunidad de madrid'},
    '14': {'murcia', 'region de murcia'},
    '15': {'navarra', 'comunidad foral de navarra'},
    '16': {'pais vasco', 'euskadi'},
    '17': {'la rioja', 'rioja'},
}



In [ ]:
def build_dim_municipalities(municipalities_path: Path, provinces_path: Path) -> pd.DataFrame:
    municipality_source = pd.read_csv(
        municipalities_path,
        dtype={'codauto': 'string', 'cpro': 'string', 'cmun': 'string', 'dc': 'string', 'nombre': 'string'},
    )
    province_source = pd.read_csv(
        provinces_path,
        dtype={'codauto': 'string', 'cpro': 'string', 'comunidad_autonoma': 'string', 'provincia': 'string'},
    )
    validate_required_columns(municipality_source, {'codauto', 'cpro', 'cmun', 'dc', 'nombre'})
    validate_required_columns(province_source, {'codauto', 'cpro', 'comunidad_autonoma', 'provincia'})

    municipality_source = municipality_source.rename(columns={'codauto': 'cauto', 'nombre': 'municipio'})
    province_source = province_source.rename(columns={'codauto': 'cauto'})

    for frame in (municipality_source, province_source):
        frame['cauto'] = normalize_ine_code_series(frame['cauto'], width=2, column_name='cauto')
        frame['cpro'] = normalize_ine_code_series(frame['cpro'], width=2, column_name='cpro')
    municipality_source['cmun'] = normalize_ine_code_series(municipality_source['cmun'], width=3, column_name='cmun')
    municipality_source['dc'] = normalize_ine_code_series(municipality_source['dc'], width=1, column_name='dc')

    if municipality_source.duplicated(['cauto', 'cpro', 'cmun']).any():
        raise ValueError('La fuente municipal contiene claves administrativas duplicadas.')
    if province_source.duplicated(['cpro']).any():
        raise ValueError('La fuente provincial contiene códigos de provincia duplicados.')

    dim = municipality_source.merge(
        province_source,
        on=['cauto', 'cpro'],
        how='left',
        validate='many_to_one',
        indicator=True,
    )
    if not dim['_merge'].eq('both').all():
        missing = dim.loc[dim['_merge'] != 'both', ['cauto', 'cpro']].drop_duplicates()
        raise ValueError(f'Hay municipios sin provincia o comunidad asociada: {missing.to_dict("records")[:10]}')
    dim = dim.drop(columns='_merge')

    dim['municipio_norm'] = dim['municipio'].map(normalize_text_or_none)
    dim['provincia_norm'] = dim['provincia'].map(normalize_text_or_none)
    dim['comunidad_autonoma_norm'] = dim['comunidad_autonoma'].map(normalize_text_or_none)
    dim['municipio_lookup_names_norm'] = dim['municipio'].map(normalized_name_variants)
    dim['ine_municipality_code'] = dim['cpro'] + dim['cmun']
    dim['ine_province_code'] = dim['cpro']
    dim['ine_autonomous_community_code'] = dim['cauto']

    required_non_null = [
        'municipio', 'municipio_norm', 'provincia', 'provincia_norm',
        'comunidad_autonoma', 'comunidad_autonoma_norm', 'ine_municipality_code',
    ]
    if dim[required_non_null].isna().any().any():
        raise ValueError('La dimensión municipal contiene nulos en campos estructurales.')
    if dim['ine_municipality_code'].duplicated().any():
        raise ValueError('El código INE municipal no es único.')
    if dim['municipio_lookup_names_norm'].map(len).eq(0).any():
        raise ValueError('Existen municipios sin nombres de búsqueda.')

    return dim.sort_values(['cpro', 'cmun']).reset_index(drop=True)



## 3. Contratos de resolución

In [ ]:
class AdministrativeUnitResolutionStatus(str, Enum):
    RESOLVED = 'resolved'
    AMBIGUOUS = 'ambiguous'
    NOT_FOUND = 'not_found'
    NOT_PROVIDED = 'not_provided'
    CONFLICT = 'conflict'


class LocationResolutionStatus(str, Enum):
    RESOLVED = 'resolved'
    PARTIALLY_RESOLVED = 'partially_resolved'
    AMBIGUOUS = 'ambiguous'
    NOT_FOUND = 'not_found'
    NOT_PROVIDED = 'not_provided'
    CONFLICT = 'conflict'


class InputAdministrativeLevel(str, Enum):
    MUNICIPALITY = 'municipality'
    PROVINCE = 'province'
    AUTONOMOUS_COMMUNITY = 'autonomous_community'


class AutonomousCommunityLocation(BaseModel):
    autonomous_community: str
    autonomous_community_norm: str
    ine_autonomous_community_code: str


class ProvinceLocation(AutonomousCommunityLocation):
    province: str
    province_norm: str
    ine_province_code: str


class MunicipalityLocation(ProvinceLocation):
    municipality: str
    municipality_norm: str
    ine_municipality_code: str


class AutonomousCommunityLookupResult(BaseModel):
    query: str | None = None
    resolution_status: AdministrativeUnitResolutionStatus
    resolved: AutonomousCommunityLocation | None = None
    candidates: list[AutonomousCommunityLocation] = Field(default_factory=list)
    matched_by: str | None = None
    reason: str | None = None


class ProvinceLookupResult(BaseModel):
    query: str | None = None
    resolution_status: AdministrativeUnitResolutionStatus
    resolved: ProvinceLocation | None = None
    candidates: list[ProvinceLocation] = Field(default_factory=list)
    matched_by: str | None = None
    reason: str | None = None


class MunicipalityLookupResult(BaseModel):
    query: str | None = None
    province_hint: str | None = None
    autonomous_community_hint: str | None = None
    resolution_status: AdministrativeUnitResolutionStatus
    resolved: MunicipalityLocation | None = None
    candidates: list[MunicipalityLocation] = Field(default_factory=list)
    matched_by: str | None = None
    reason: str | None = None



### 3.1. Generar y validar dimensiones derivadas

In [ ]:
municipalities = build_dim_municipalities(
    INE_MUNICIPALITIES_SOURCE_PATH,
    INE_PROVINCES_SOURCE_PATH,
)

save_parquet(
    municipalities,
    DIM_MUNICIPALITIES_PATH,
)
provinces = (
    municipalities[['cauto', 'cpro', 'provincia', 'provincia_norm', 'comunidad_autonoma', 'comunidad_autonoma_norm']]
    .drop_duplicates('cpro')
    .reset_index(drop=True)
)
autonomous_communities = (
    municipalities[['cauto', 'comunidad_autonoma', 'comunidad_autonoma_norm']]
    .drop_duplicates('cauto')
    .reset_index(drop=True)
)
provinces['province_lookup_names_norm'] = provinces.apply(
    lambda row: normalized_name_variants(row['provincia'], aliases=PROVINCE_ALIASES_BY_CODE.get(row['cpro'], set())), axis=1
)
autonomous_communities['autonomous_community_lookup_names_norm'] = autonomous_communities.apply(
    lambda row: normalized_name_variants(row['comunidad_autonoma'], aliases=AUTONOMOUS_COMMUNITY_ALIASES_BY_CODE.get(row['cauto'], set())), axis=1
)

print(f"Municipios: {len(municipalities):,}")
print(f"Provincias: {len(provinces):,}")
print(f"Comunidades autónomas y ciudades autónomas: {len(autonomous_communities):,}")

display(
    municipalities.loc[
        municipalities["municipio_norm"].str.contains(
            "pontes|cedeira|andevalo|puebla de guzman",
            na=False,
        ),
        [
            "ine_municipality_code",
            "municipio",
            "provincia",
            "comunidad_autonoma",
            "municipio_lookup_names_norm",
        ],
    ]
)


## 4. Cargar y validar las localizaciones extraídas

In [ ]:
project_locations = pd.read_parquet(PROJECT_LOCATIONS_PATH).reset_index(drop=True)

REQUIRED_PROJECT_LOCATION_COLUMNS = {
    "project_location_id",
    "project_mention_id",
    "event_id",
    "identificador_boe",
    "fecha_publicacion",
    "location_index",
    "municipality_raw",
    "municipality_raw_norm",
    "province_hint_raw",
    "province_hint_raw_norm",
    "autonomous_community_hint_raw",
    "autonomous_community_hint_raw_norm",
    "location_evidence",
}

validate_required_columns(project_locations, REQUIRED_PROJECT_LOCATION_COLUMNS)

if project_locations["project_location_id"].duplicated().any():
    duplicates = project_locations.loc[
        project_locations["project_location_id"].duplicated(keep=False),
        "project_location_id",
    ].tolist()
    raise ValueError(
        "project_location_id contiene duplicados: "
        f"{duplicates[:10]}"
    )

for raw_column, normalized_column in [
    ("municipality_raw", "municipality_raw_norm"),
    ("province_hint_raw", "province_hint_raw_norm"),
    (
        "autonomous_community_hint_raw",
        "autonomous_community_hint_raw_norm",
    ),
]:
    expected = project_locations[raw_column].map(normalize_text_or_none)
    actual = project_locations[normalized_column]
    mismatch = expected.fillna("<NA>") != actual.fillna("<NA>")

    if mismatch.any():
        examples = project_locations.loc[
            mismatch,
            [raw_column, normalized_column],
        ].head(10)
        raise ValueError(
            f"{normalized_column} no coincide con la normalización de "
            f"{raw_column}: {examples.to_dict('records')}"
        )

print(f"Localizaciones de entrada: {len(project_locations):,}")
display(project_locations.head())


## 5. Reglas de comparación

La resolución se aplica en este orden:

1. denominación o alias normalizado;
2. coincidencia determinista de tokens;
3. similitud ortográfica conservadora, solo si existe un candidato único por encima del umbral y separado del segundo candidato.

La similitud ortográfica no se utiliza para inferir el nivel administrativo de la mención. Solo interviene en la resolución municipal final.


Las primitivas genéricas `text_tokens` y `token_overlap_score` se reutilizan desde `renewables_permitting.text_matching`; las reglas administrativas permanecen en este notebook.


In [ ]:
STOP_TOKENS = {
    'de', 'del', 'd', 'da', 'das', 'do', 'dos', 'dels', 'deth', 'dera', 'des',
    'en', 'y', 'e', 'i', 'eta', 'la', 'las', 'el', 'los',
    'ayuntamiento', 'municipio', 'municipios', 'municipal', 'termino', 'terminos',
    'concello', 'concellos', 'termo', 'ajuntament', 'municipi', 'terme',
    'udal', 'udala', 'udalerri', 'udalerria',
}
ADMINISTRATIVE_NAME_STOP_TOKENS = {
    'de', 'del', 'la', 'las', 'el', 'los', 'y', 'e',
    'comunidad', 'comunitat', 'region', 'principado',
}
INVALID_MUNICIPALITY_VALUES = {'no consta', 'desconocido', 'no aplica', 'ninguno'}


def coerce_lookup_names(value: Any) -> list[Any]:
    if value is None:
        return []
    if isinstance(value, str):
        return [value]
    if isinstance(value, (list, tuple, set)):
        return list(value)
    if hasattr(value, 'tolist'):
        converted = value.tolist()
        return converted if isinstance(converted, list) else [converted]
    if is_null_like(value):
        return []
    return [value]


def lookup_names_contain(lookup_names: Any, query_norm: str | None) -> bool:
    if query_norm is None:
        return False
    names = {normalize_text_or_none(value) for value in coerce_lookup_names(lookup_names)}
    return query_norm in names


def administrative_name_token_matches(query: Any, candidate: Any) -> bool:
    q = text_tokens(query, stop_tokens=ADMINISTRATIVE_NAME_STOP_TOKENS)
    c = text_tokens(candidate, stop_tokens=ADMINISTRATIVE_NAME_STOP_TOKENS)
    return bool(q) and q == c


def lookup_names_token_match(lookup_names: Any, query: Any) -> bool:
    return any(administrative_name_token_matches(query, value) for value in coerce_lookup_names(lookup_names))


def municipality_token_matches(query: Any, lookup_names: Any, *, allow_single_token: bool) -> bool:
    q = text_tokens(query, stop_tokens=STOP_TOKENS)
    if not q:
        return False
    for candidate in coerce_lookup_names(lookup_names):
        c = text_tokens(candidate, stop_tokens=STOP_TOKENS)
        if not c:
            continue
        if q == c:
            return True
        if q.issubset(c):
            if len(q) >= 2:
                return True
            token = next(iter(q))
            if allow_single_token and len(token) >= 5:
                return True
        if len(q) >= 2 and token_overlap_score(query, candidate, stop_tokens=STOP_TOKENS) >= 0.8:
            return True
    return False


def fuzzy_similarity(query: Any, lookup_names: Any) -> float:
    q = normalize_text_or_none(query)
    if q is None or len(q.replace(' ', '')) < 5:
        return 0.0
    return max(
        (SequenceMatcher(None, q, normalize_text_or_none(candidate) or '').ratio() for candidate in coerce_lookup_names(lookup_names)),
        default=0.0,
    )



In [ ]:
def row_to_autonomous_community_location(row: pd.Series) -> AutonomousCommunityLocation:
    return AutonomousCommunityLocation(
        autonomous_community=row['comunidad_autonoma'],
        autonomous_community_norm=row['comunidad_autonoma_norm'],
        ine_autonomous_community_code=row['cauto'],
    )


def row_to_province_location(row: pd.Series) -> ProvinceLocation:
    return ProvinceLocation(
        province=row['provincia'], province_norm=row['provincia_norm'], ine_province_code=row['cpro'],
        autonomous_community=row['comunidad_autonoma'], autonomous_community_norm=row['comunidad_autonoma_norm'],
        ine_autonomous_community_code=row['cauto'],
    )


def row_to_municipality_location(row: pd.Series) -> MunicipalityLocation:
    return MunicipalityLocation(
        municipality=row['municipio'], municipality_norm=row['municipio_norm'], ine_municipality_code=row['ine_municipality_code'],
        province=row['provincia'], province_norm=row['provincia_norm'], ine_province_code=row['cpro'],
        autonomous_community=row['comunidad_autonoma'], autonomous_community_norm=row['comunidad_autonoma_norm'],
        ine_autonomous_community_code=row['cauto'],
    )


def build_ac_result(query, matches, *, matched_by, reason, empty_status):
    matches = matches.drop_duplicates('cauto')
    if matches.empty:
        return AutonomousCommunityLookupResult(query=query, resolution_status=empty_status, matched_by=matched_by, reason=reason)
    if len(matches) == 1:
        return AutonomousCommunityLookupResult(query=query, resolution_status=AdministrativeUnitResolutionStatus.RESOLVED, resolved=row_to_autonomous_community_location(matches.iloc[0]), matched_by=matched_by, reason=reason)
    return AutonomousCommunityLookupResult(query=query, resolution_status=AdministrativeUnitResolutionStatus.AMBIGUOUS, candidates=[row_to_autonomous_community_location(row) for _, row in matches.iterrows()], matched_by=matched_by, reason='La mención coincide con varias comunidades autónomas.')


def build_province_result(query, matches, *, matched_by, reason, empty_status):
    matches = matches.drop_duplicates('cpro')
    if matches.empty:
        return ProvinceLookupResult(query=query, resolution_status=empty_status, matched_by=matched_by, reason=reason)
    if len(matches) == 1:
        return ProvinceLookupResult(query=query, resolution_status=AdministrativeUnitResolutionStatus.RESOLVED, resolved=row_to_province_location(matches.iloc[0]), matched_by=matched_by, reason=reason)
    return ProvinceLookupResult(query=query, resolution_status=AdministrativeUnitResolutionStatus.AMBIGUOUS, candidates=[row_to_province_location(row) for _, row in matches.iterrows()], matched_by=matched_by, reason='La mención coincide con varias provincias.')


def build_municipality_result(query, province_hint, ac_hint, matches, *, matched_by, reason, empty_status):
    matches = matches.drop_duplicates(['cauto', 'cpro', 'cmun'])
    if matches.empty:
        return MunicipalityLookupResult(query=query, province_hint=province_hint, autonomous_community_hint=ac_hint, resolution_status=empty_status, matched_by=matched_by, reason=reason)
    if len(matches) == 1:
        return MunicipalityLookupResult(query=query, province_hint=province_hint, autonomous_community_hint=ac_hint, resolution_status=AdministrativeUnitResolutionStatus.RESOLVED, resolved=row_to_municipality_location(matches.iloc[0]), matched_by=matched_by, reason=reason)
    return MunicipalityLookupResult(query=query, province_hint=province_hint, autonomous_community_hint=ac_hint, resolution_status=AdministrativeUnitResolutionStatus.AMBIGUOUS, candidates=[row_to_municipality_location(row) for _, row in matches.iterrows()], matched_by=matched_by, reason='Existen varias coincidencias municipales compatibles con los criterios disponibles.')


def resolve_autonomous_community(query: Any) -> AutonomousCommunityLookupResult:
    qn = normalize_text_or_none(query)
    if qn is None:
        return build_ac_result(query, autonomous_communities.iloc[0:0], matched_by=None, reason='No se proporcionó una comunidad autónoma.', empty_status=AdministrativeUnitResolutionStatus.NOT_PROVIDED)
    exact = autonomous_communities.loc[autonomous_communities['autonomous_community_lookup_names_norm'].map(lambda names: lookup_names_contain(names, qn))]
    if not exact.empty:
        return build_ac_result(query, exact, matched_by='autonomous_community_lookup_name', reason='Comunidad autónoma resuelta por denominación o alias normalizado.', empty_status=AdministrativeUnitResolutionStatus.NOT_FOUND)
    token = autonomous_communities.loc[autonomous_communities['autonomous_community_lookup_names_norm'].map(lambda names: lookup_names_token_match(names, query))]
    return build_ac_result(query, token, matched_by='autonomous_community_token' if not token.empty else 'not_found', reason='Comunidad autónoma resuelta por igualdad estricta de tokens.' if not token.empty else 'La comunidad autónoma no existe en la dimensión INE.', empty_status=AdministrativeUnitResolutionStatus.NOT_FOUND)


def resolve_province(query: Any) -> ProvinceLookupResult:
    qn = normalize_text_or_none(query)
    if qn is None:
        return build_province_result(query, provinces.iloc[0:0], matched_by=None, reason='No se proporcionó una provincia.', empty_status=AdministrativeUnitResolutionStatus.NOT_PROVIDED)
    exact = provinces.loc[provinces['province_lookup_names_norm'].map(lambda names: lookup_names_contain(names, qn))]
    if not exact.empty:
        return build_province_result(query, exact, matched_by='province_lookup_name', reason='Provincia resuelta por denominación o alias normalizado.', empty_status=AdministrativeUnitResolutionStatus.NOT_FOUND)
    token = provinces.loc[provinces['province_lookup_names_norm'].map(lambda names: lookup_names_token_match(names, query))]
    return build_province_result(query, token, matched_by='province_token' if not token.empty else 'not_found', reason='Provincia resuelta por igualdad estricta de tokens.' if not token.empty else 'La provincia no existe en la dimensión INE.', empty_status=AdministrativeUnitResolutionStatus.NOT_FOUND)


def constrain_matches(matches: pd.DataFrame, province_code: str | None, ac_code: str | None) -> tuple[pd.DataFrame, list[str]]:
    if matches.empty:
        return matches, []
    constrained = matches
    criteria: list[str] = []
    if province_code is not None:
        constrained = constrained.loc[constrained['cpro'] == province_code]
        criteria.append('province')
    if ac_code is not None:
        constrained = constrained.loc[constrained['cauto'] == ac_code]
        criteria.append('autonomous_community')
    if constrained.empty:
        return matches, []
    return constrained, criteria


def resolve_municipality(
    query: Any,
    province_hint: Any = None,
    ac_hint: Any = None,
    *,
    province_code: str | None = None,
    ac_code: str | None = None,
    allow_fuzzy: bool = True,
) -> MunicipalityLookupResult:
    qn = normalize_text_or_none(query)
    if qn is None or qn in INVALID_MUNICIPALITY_VALUES:
        return build_municipality_result(query, province_hint, ac_hint, municipalities.iloc[0:0], matched_by=None, reason='No se proporcionó una mención municipal resoluble.', empty_status=AdministrativeUnitResolutionStatus.NOT_PROVIDED)

    exact = municipalities.loc[municipalities['municipio_lookup_names_norm'].map(lambda names: lookup_names_contain(names, qn))]
    if not exact.empty:
        constrained, criteria = constrain_matches(exact, province_code, ac_code)
        suffix = '_and_' + '_and_'.join(criteria) if criteria else ''
        return build_municipality_result(query, province_hint, ac_hint, constrained, matched_by=f'municipality_lookup_name{suffix}', reason='Municipio resuelto por nombre o variante normalizada' + (f' y restringido por {" y ".join(criteria)}' if criteria else '') + '.', empty_status=AdministrativeUnitResolutionStatus.NOT_FOUND)

    search_space = municipalities
    active: list[str] = []
    if province_code is not None:
        search_space = search_space.loc[search_space['cpro'] == province_code]
        active.append('province')
    elif ac_code is not None:
        search_space = search_space.loc[search_space['cauto'] == ac_code]
        active.append('autonomous_community')

    token = search_space.loc[search_space['municipio_lookup_names_norm'].map(lambda names: municipality_token_matches(query, names, allow_single_token=bool(active)))]
    if token.empty and active:
        global_token = municipalities.loc[municipalities['municipio_lookup_names_norm'].map(lambda names: municipality_token_matches(query, names, allow_single_token=False))]
        if not global_token.empty:
            token = global_token
            active = []
    if not token.empty:
        suffix = '_and_' + '_and_'.join(active) if active else ''
        return build_municipality_result(query, province_hint, ac_hint, token, matched_by=f'municipality_token{suffix}', reason='Municipio resuelto por coincidencia determinista de tokens' + (f' restringida por {" y ".join(active)}' if active else '') + '.', empty_status=AdministrativeUnitResolutionStatus.NOT_FOUND)

    if not allow_fuzzy:
        return build_municipality_result(
            query,
            province_hint,
            ac_hint,
            municipalities.iloc[0:0],
            matched_by="not_found",
            reason=(
                "No existe coincidencia municipal por nombre ni por tokens "
                "en la dimensión INE."
            ),
            empty_status=AdministrativeUnitResolutionStatus.NOT_FOUND,
        )

    # Corrección tipográfica conservadora: solo si hay un mejor candidato único.
    fuzzy_space = search_space if not search_space.empty else municipalities
    scores = fuzzy_space['municipio_lookup_names_norm'].map(lambda names: fuzzy_similarity(query, names))
    threshold = 0.90 if active else 0.94
    margin = 0.04
    if len(scores):
        ranked = scores.sort_values(ascending=False)
        best_score = float(ranked.iloc[0])
        second_score = float(ranked.iloc[1]) if len(ranked) > 1 else 0.0
        if best_score >= threshold and best_score - second_score >= margin:
            best = fuzzy_space.loc[[ranked.index[0]]]
            suffix = '_and_' + '_and_'.join(active) if active else ''
            return build_municipality_result(query, province_hint, ac_hint, best, matched_by=f'municipality_fuzzy{suffix}', reason=f'Municipio resuelto por similitud ortográfica conservadora ({best_score:.3f})' + (f' restringida por {" y ".join(active)}' if active else '') + '.', empty_status=AdministrativeUnitResolutionStatus.NOT_FOUND)
        if best_score >= threshold:
            near_best = fuzzy_space.loc[scores >= best_score - 0.01]
            return build_municipality_result(query, province_hint, ac_hint, near_best, matched_by='municipality_fuzzy_ambiguous', reason='La similitud ortográfica no permite seleccionar un candidato único con margen suficiente.', empty_status=AdministrativeUnitResolutionStatus.NOT_FOUND)

    return build_municipality_result(query, province_hint, ac_hint, municipalities.iloc[0:0], matched_by='not_found', reason='No existe coincidencia municipal por nombre, tokens ni similitud ortográfica conservadora en la dimensión INE.', empty_status=AdministrativeUnitResolutionStatus.NOT_FOUND)



## 6. Inferencia del nivel y reconciliación jerárquica

Una cadena almacenada en `municipality_raw` puede corresponder realmente a una provincia o comunidad autónoma. La inferencia utiliza la evidencia textual y los campos de pista, pero conserva por defecto la semántica municipal cuando no hay prueba suficiente para reclasificarla.


In [ ]:
def direct_labeled_name(evidence: Any, label_pattern: str, raw_name: Any) -> bool:
    evidence_norm = normalize_text_or_none(evidence)
    raw_norm = normalize_text_or_none(raw_name)
    if not evidence_norm or not raw_norm:
        return False
    pattern = rf'\b(?:{label_pattern})\s+(?:de\s+|del\s+|da\s+|do\s+|d[ae]s\s+)?{re.escape(raw_norm)}\b'
    return re.search(pattern, evidence_norm) is not None


def infer_input_level(row: pd.Series) -> tuple[InputAdministrativeLevel, str, str]:
    raw = row.get('municipality_raw')
    raw_norm = normalize_text_or_none(raw)
    province_hint_norm = normalize_text_or_none(row.get('province_hint_raw'))
    ac_hint_norm = normalize_text_or_none(row.get('autonomous_community_hint_raw'))

    raw_municipality = resolve_municipality(raw, allow_fuzzy=False)
    raw_province = resolve_province(raw)
    raw_ac = resolve_autonomous_community(raw)

    municipality_resolved = raw_municipality.resolution_status == AdministrativeUnitResolutionStatus.RESOLVED
    province_resolved = raw_province.resolution_status == AdministrativeUnitResolutionStatus.RESOLVED
    ac_resolved = raw_ac.resolution_status == AdministrativeUnitResolutionStatus.RESOLVED

    direct_municipality = direct_labeled_name(row.get('location_evidence'), r'municipio|termino municipal|concello|termo municipal', raw)
    direct_province = direct_labeled_name(row.get('location_evidence'), r'provincia', raw)
    direct_ac = direct_labeled_name(row.get('location_evidence'), r'comunidad autonoma|comunitat autonoma|region', raw)

    if direct_municipality and municipality_resolved:
        return InputAdministrativeLevel.MUNICIPALITY, 'evidence_municipality_label', 'La evidencia etiqueta directamente la mención como municipio.'
    if direct_province and province_resolved and not direct_municipality:
        return InputAdministrativeLevel.PROVINCE, 'evidence_province_label', 'La evidencia etiqueta directamente la mención como provincia.'
    if direct_ac and ac_resolved and not direct_municipality:
        return InputAdministrativeLevel.AUTONOMOUS_COMMUNITY, 'evidence_autonomous_community_label', 'La evidencia etiqueta directamente la mención como comunidad autónoma.'

    if ac_resolved and raw_norm == ac_hint_norm and not municipality_resolved:
        return InputAdministrativeLevel.AUTONOMOUS_COMMUNITY, 'matches_autonomous_community_hint', 'La mención coincide con la comunidad autónoma explícita y no con un municipio.'
    if province_resolved and raw_norm == province_hint_norm and not municipality_resolved:
        return InputAdministrativeLevel.PROVINCE, 'matches_province_hint', 'La mención coincide con la provincia explícita y no con un municipio.'
    if province_resolved and not municipality_resolved and not ac_resolved:
        return InputAdministrativeLevel.PROVINCE, 'province_only_match', 'La mención solo coincide con una provincia.'
    if ac_resolved and not municipality_resolved and not province_resolved:
        return InputAdministrativeLevel.AUTONOMOUS_COMMUNITY, 'autonomous_community_only_match', 'La mención solo coincide con una comunidad autónoma.'

    # La columna de entrada tiene semántica municipal; ante homónimos sin prueba suficiente,
    # se conserva el nivel municipal y la reconciliación posterior detecta conflictos.
    return InputAdministrativeLevel.MUNICIPALITY, 'municipality_schema_default', 'No existe evidencia suficiente para reclasificar la mención a un nivel superior.'


def not_provided_municipality(query: Any, level: InputAdministrativeLevel) -> MunicipalityLookupResult:
    return MunicipalityLookupResult(
        query=query,
        resolution_status=AdministrativeUnitResolutionStatus.NOT_PROVIDED,
        matched_by=f'reclassified_as_{level.value}',
        reason=f'La mención se ha reclasificado como {level.value}; no contiene un municipio.',
    )


def not_provided_province(query: Any, level: InputAdministrativeLevel) -> ProvinceLookupResult:
    return ProvinceLookupResult(
        query=query,
        resolution_status=AdministrativeUnitResolutionStatus.NOT_PROVIDED,
        matched_by=f'reclassified_as_{level.value}',
        reason=f'La mención se ha reclasificado como {level.value}; no contiene una provincia.',
    )


def inherited_province_result(canonical: ProvinceLocation, *, sources: list[str]) -> ProvinceLookupResult:
    return ProvinceLookupResult(
        resolution_status=AdministrativeUnitResolutionStatus.RESOLVED,
        resolved=canonical,
        matched_by='inherited_from_publication_context',
        reason='Provincia heredada del contexto único y compatible de la misma mención de proyecto dentro de la publicación: ' + ', '.join(sorted(sources)) + '.',
    )


def inherited_ac_result(canonical: AutonomousCommunityLocation, *, sources: list[str]) -> AutonomousCommunityLookupResult:
    return AutonomousCommunityLookupResult(
        resolution_status=AdministrativeUnitResolutionStatus.RESOLVED,
        resolved=canonical,
        matched_by='inherited_from_publication_context',
        reason='Comunidad autónoma heredada del contexto único y compatible de la misma mención de proyecto dentro de la publicación: ' + ', '.join(sorted(sources)) + '.',
    )


def canonical_province_from_municipality(m: MunicipalityLocation) -> ProvinceLocation:
    return ProvinceLocation(**m.model_dump(include={'province', 'province_norm', 'ine_province_code', 'autonomous_community', 'autonomous_community_norm', 'ine_autonomous_community_code'}))


def canonical_ac_from_province(p: ProvinceLocation) -> AutonomousCommunityLocation:
    return AutonomousCommunityLocation(**p.model_dump(include={'autonomous_community', 'autonomous_community_norm', 'ine_autonomous_community_code'}))


def reconcile_province(lookup: ProvinceLookupResult, canonical: ProvinceLocation, *, source: str) -> ProvinceLookupResult:
    if lookup.resolution_status == AdministrativeUnitResolutionStatus.RESOLVED and lookup.resolved.ine_province_code != canonical.ine_province_code:
        return ProvinceLookupResult(query=lookup.query, resolution_status=AdministrativeUnitResolutionStatus.CONFLICT, resolved=canonical, candidates=[lookup.resolved], matched_by=f'conflict_with_{source}', reason=f'La provincia disponible contradice la determinada por {source}; se conserva la jerarquía de la unidad más específica.')
    if lookup.resolution_status == AdministrativeUnitResolutionStatus.RESOLVED:
        return ProvinceLookupResult(query=lookup.query, resolution_status=AdministrativeUnitResolutionStatus.RESOLVED, resolved=canonical, matched_by=f'{lookup.matched_by}_validated_by_{source}', reason=f'La provincia disponible coincide con la determinada por {source}.')
    return ProvinceLookupResult(query=lookup.query, resolution_status=AdministrativeUnitResolutionStatus.RESOLVED, resolved=canonical, matched_by=f'derived_from_{source}', reason=f'Provincia derivada de {source}.')


def reconcile_ac(lookup: AutonomousCommunityLookupResult, canonical: AutonomousCommunityLocation, *, source: str) -> AutonomousCommunityLookupResult:
    if lookup.resolution_status == AdministrativeUnitResolutionStatus.RESOLVED and lookup.resolved.ine_autonomous_community_code != canonical.ine_autonomous_community_code:
        return AutonomousCommunityLookupResult(query=lookup.query, resolution_status=AdministrativeUnitResolutionStatus.CONFLICT, resolved=canonical, candidates=[lookup.resolved], matched_by=f'conflict_with_{source}', reason=f'La comunidad autónoma disponible contradice la determinada por {source}; se conserva la jerarquía de la unidad más específica.')
    if lookup.resolution_status == AdministrativeUnitResolutionStatus.RESOLVED:
        return AutonomousCommunityLookupResult(query=lookup.query, resolution_status=AdministrativeUnitResolutionStatus.RESOLVED, resolved=canonical, matched_by=f'{lookup.matched_by}_validated_by_{source}', reason=f'La comunidad autónoma disponible coincide con la determinada por {source}.')
    return AutonomousCommunityLookupResult(query=lookup.query, resolution_status=AdministrativeUnitResolutionStatus.RESOLVED, resolved=canonical, matched_by=f'derived_from_{source}', reason=f'Comunidad autónoma derivada de {source}.')



## 7. Contexto administrativo interno por publicación

El contexto se calcula de forma temporal para cada clave compuesta `identificador_boe + event_id + project_mention_id`.

Esta delimitación garantiza que:

- no se herede información entre publicaciones diferentes;
- no se mezclen proyectos distintos contenidos en una misma publicación;
- una provincia o comunidad autónoma solo se propague cuando sea única y compatible dentro de esa mención publicada.

El contexto se incorpora como metadatos de trazabilidad en cada fila resuelta, pero no se guarda como un dataset independiente.


In [ ]:
CONTEXT_KEY_COLUMNS = (
    "identificador_boe",
    "event_id",
    "project_mention_id",
)


@dataclass(frozen=True)
class PublicationProjectAdministrativeContext:
    identificador_boe: str
    event_id: str
    project_mention_id: str
    province: ProvinceLocation | None
    province_sources: tuple[str, ...]
    province_candidate_codes: tuple[str, ...]
    autonomous_community: AutonomousCommunityLocation | None
    autonomous_community_sources: tuple[str, ...]
    autonomous_community_candidate_codes: tuple[str, ...]
    status: str
    reason: str


def publication_context_key(row: pd.Series) -> tuple[str, str, str]:
    return tuple(str(row[column]) for column in CONTEXT_KEY_COLUMNS)


def build_publication_contexts(
    project_locations: pd.DataFrame,
) -> tuple[
    dict[tuple[str, str, str], PublicationProjectAdministrativeContext],
    pd.DataFrame,
]:
    required = {
        *CONTEXT_KEY_COLUMNS,
        "municipality_raw",
        "province_hint_raw",
        "autonomous_community_hint_raw",
        "location_evidence",
    }
    validate_required_columns(project_locations, required)
    preliminary: list[dict[str, Any]] = []

    for idx, row in project_locations.iterrows():
        level, level_by, level_reason = infer_input_level(row)
        province_hint = resolve_province(row.get("province_hint_raw"))
        ac_hint = resolve_autonomous_community(
            row.get("autonomous_community_hint_raw")
        )

        province_code = (
            province_hint.resolved.ine_province_code
            if province_hint.resolution_status
            == AdministrativeUnitResolutionStatus.RESOLVED
            else None
        )
        ac_code = (
            ac_hint.resolved.ine_autonomous_community_code
            if ac_hint.resolution_status
            == AdministrativeUnitResolutionStatus.RESOLVED
            else None
        )

        first_pass_municipality = (
            resolve_municipality(
                row.get("municipality_raw"),
                row.get("province_hint_raw"),
                row.get("autonomous_community_hint_raw"),
                province_code=province_code,
                ac_code=ac_code,
            )
            if level == InputAdministrativeLevel.MUNICIPALITY
            else None
        )
        raw_province = (
            resolve_province(row.get("municipality_raw"))
            if level == InputAdministrativeLevel.PROVINCE
            else None
        )
        raw_ac = (
            resolve_autonomous_community(row.get("municipality_raw"))
            if level == InputAdministrativeLevel.AUTONOMOUS_COMMUNITY
            else None
        )

        preliminary.append(
            {
                "_row_index": idx,
                "identificador_boe": row["identificador_boe"],
                "event_id": row["event_id"],
                "project_mention_id": row["project_mention_id"],
                "input_level": level,
                "input_level_matched_by": level_by,
                "input_level_reason": level_reason,
                "province_hint_lookup": province_hint,
                "ac_hint_lookup": ac_hint,
                "first_pass_municipality": first_pass_municipality,
                "raw_province_lookup": raw_province,
                "raw_ac_lookup": raw_ac,
            }
        )

    pre_df = pd.DataFrame(preliminary).set_index("_row_index", drop=False)
    contexts: dict[
        tuple[str, str, str],
        PublicationProjectAdministrativeContext,
    ] = {}

    for context_values, group in pre_df.groupby(
        list(CONTEXT_KEY_COLUMNS),
        sort=False,
        dropna=False,
    ):
        context_key = tuple(str(value) for value in context_values)
        identificador_boe, event_id, project_mention_id = context_key

        province_items: dict[str, dict[str, Any]] = {}
        ac_items: dict[str, dict[str, Any]] = {}

        def add_ac(
            location: AutonomousCommunityLocation,
            source: str,
        ) -> None:
            item = ac_items.setdefault(
                location.ine_autonomous_community_code,
                {"location": location, "sources": set()},
            )
            item["sources"].add(source)

        def add_province(
            location: ProvinceLocation,
            source: str,
        ) -> None:
            item = province_items.setdefault(
                location.ine_province_code,
                {"location": location, "sources": set()},
            )
            item["sources"].add(source)
            add_ac(
                canonical_ac_from_province(location),
                f"province:{source}",
            )

        for _, item in group.iterrows():
            province_hint = item["province_hint_lookup"]
            if (
                province_hint.resolution_status
                == AdministrativeUnitResolutionStatus.RESOLVED
            ):
                add_province(
                    province_hint.resolved,
                    "explicit_province_hint",
                )

            ac_hint = item["ac_hint_lookup"]
            if (
                ac_hint.resolution_status
                == AdministrativeUnitResolutionStatus.RESOLVED
            ):
                add_ac(
                    ac_hint.resolved,
                    "explicit_autonomous_community_hint",
                )

            raw_province = item["raw_province_lookup"]
            if (
                raw_province is not None
                and raw_province.resolution_status
                == AdministrativeUnitResolutionStatus.RESOLVED
            ):
                add_province(
                    raw_province.resolved,
                    "reclassified_province_row",
                )

            raw_ac = item["raw_ac_lookup"]
            if (
                raw_ac is not None
                and raw_ac.resolution_status
                == AdministrativeUnitResolutionStatus.RESOLVED
            ):
                add_ac(
                    raw_ac.resolved,
                    "reclassified_autonomous_community_row",
                )

            first_pass_municipality = item["first_pass_municipality"]
            if (
                first_pass_municipality is not None
                and first_pass_municipality.resolution_status
                == AdministrativeUnitResolutionStatus.RESOLVED
            ):
                add_province(
                    canonical_province_from_municipality(
                        first_pass_municipality.resolved
                    ),
                    "resolved_municipality_row",
                )

        province_codes = tuple(sorted(province_items))
        ac_codes = tuple(sorted(ac_items))

        province = (
            province_items[province_codes[0]]["location"]
            if len(province_codes) == 1
            else None
        )
        province_sources = (
            tuple(
                sorted(
                    province_items[province_codes[0]]["sources"]
                )
            )
            if province is not None
            else tuple(
                sorted(
                    {
                        source
                        for item in province_items.values()
                        for source in item["sources"]
                    }
                )
            )
        )

        autonomous_community = (
            ac_items[ac_codes[0]]["location"]
            if len(ac_codes) == 1
            else None
        )
        autonomous_community_sources = (
            tuple(sorted(ac_items[ac_codes[0]]["sources"]))
            if autonomous_community is not None
            else tuple(
                sorted(
                    {
                        source
                        for item in ac_items.values()
                        for source in item["sources"]
                    }
                )
            )
        )

        if len(province_codes) > 1:
            status = "ambiguous"
            reason = (
                "La misma mención de proyecto dentro de la publicación "
                f"contiene varias provincias candidatas: "
                f"{list(province_codes)}; no se propaga una provincia común."
            )
        elif len(ac_codes) > 1:
            status = "conflict"
            reason = (
                "La misma mención de proyecto dentro de la publicación "
                f"contiene comunidades autónomas incompatibles: "
                f"{list(ac_codes)}; no se propaga una comunidad común."
            )
        elif province is not None or autonomous_community is not None:
            status = "resolved"
            reason = (
                "La misma mención de proyecto dentro de la publicación "
                "tiene un contexto administrativo único y compatible, "
                "apto para completar filas sin pista explícita."
            )
        else:
            status = "not_found"
            reason = (
                "No se pudo construir un contexto administrativo común "
                "para esta mención de proyecto dentro de la publicación."
            )

        contexts[context_key] = PublicationProjectAdministrativeContext(
            identificador_boe=identificador_boe,
            event_id=event_id,
            project_mention_id=project_mention_id,
            province=province,
            province_sources=province_sources,
            province_candidate_codes=province_codes,
            autonomous_community=autonomous_community,
            autonomous_community_sources=autonomous_community_sources,
            autonomous_community_candidate_codes=ac_codes,
            status=status,
            reason=reason,
        )

    return contexts, pre_df


## 8. Resolver localizaciones

La función principal calcula internamente el contexto acotado a la publicación y devuelve exclusivamente la tabla de localizaciones resueltas.


In [ ]:
CANONICAL_LOCATION_COLUMNS = {
    'municipality': None, 'municipality_norm': None, 'ine_municipality_code': None,
    'province': None, 'province_norm': None, 'ine_province_code': None,
    'autonomous_community': None, 'autonomous_community_norm': None, 'ine_autonomous_community_code': None,
}


def add_metadata(record: dict[str, Any], prefix: str, result: Any) -> None:
    record[f'{prefix}_resolution_status'] = result.resolution_status.value
    record[f'{prefix}_resolution_matched_by'] = result.matched_by
    record[f'{prefix}_resolution_reason'] = result.reason
    record[f"{prefix}_resolution_candidate_codes"] = [
        getattr(candidate, f"ine_{prefix}_code")
        for candidate in result.candidates
    ]


def aggregate_unit_resolution_statuses(
    statuses: tuple[
        AdministrativeUnitResolutionStatus,
        AdministrativeUnitResolutionStatus,
        AdministrativeUnitResolutionStatus,
    ],
) -> LocationResolutionStatus:
    """Agrega los tres estados administrativos por completitud jerárquica.

    La prioridad es:

    1. `conflict`;
    2. `ambiguous`;
    3. número de niveles `resolved`;
    4. ausencia (`not_found` o `not_provided`).

    `resolved` exige que municipio, provincia y comunidad autónoma estén
    resueltos. Uno o dos niveles resueltos producen `partially_resolved`.
    """
    if AdministrativeUnitResolutionStatus.CONFLICT in statuses:
        return LocationResolutionStatus.CONFLICT

    if AdministrativeUnitResolutionStatus.AMBIGUOUS in statuses:
        return LocationResolutionStatus.AMBIGUOUS

    resolved_count = sum(
        status == AdministrativeUnitResolutionStatus.RESOLVED
        for status in statuses
    )

    if resolved_count == 3:
        return LocationResolutionStatus.RESOLVED

    if resolved_count in {1, 2}:
        return LocationResolutionStatus.PARTIALLY_RESOLVED

    if AdministrativeUnitResolutionStatus.NOT_FOUND in statuses:
        return LocationResolutionStatus.NOT_FOUND

    return LocationResolutionStatus.NOT_PROVIDED


def aggregate_location_status(
    municipality: MunicipalityLookupResult,
    province: ProvinceLookupResult,
    ac: AutonomousCommunityLookupResult,
) -> LocationResolutionStatus:
    """Calcula el estado global a partir de los tres resultados individuales."""
    return aggregate_unit_resolution_statuses(
        (
            municipality.resolution_status,
            province.resolution_status,
            ac.resolution_status,
        )
    )


def build_location_resolution_reason(
    status: LocationResolutionStatus,
    municipality: MunicipalityLookupResult,
    province: ProvinceLookupResult,
    ac: AutonomousCommunityLookupResult,
) -> str:
    """Describe de forma trazable la agregación jerárquica."""
    unit_statuses = {
        "municipality": municipality.resolution_status.value,
        "province": province.resolution_status.value,
        "autonomous_community": ac.resolution_status.value,
    }
    resolved_count = sum(
        value == AdministrativeUnitResolutionStatus.RESOLVED.value
        for value in unit_statuses.values()
    )

    return (
        "Estado global de completitud jerárquica: "
        f"{resolved_count}/3 niveles resueltos; "
        f"estados individuales={unit_statuses}; "
        f"resultado={status.value}. "
        "Los conflictos y las ambigüedades tienen prioridad sobre el recuento "
        "de niveles resueltos."
    )


def resolved_level(municipality, province, ac) -> str | None:
    if municipality.resolution_status == AdministrativeUnitResolutionStatus.RESOLVED:
        return InputAdministrativeLevel.MUNICIPALITY.value
    if province.resolution_status == AdministrativeUnitResolutionStatus.RESOLVED:
        return InputAdministrativeLevel.PROVINCE.value
    if ac.resolution_status == AdministrativeUnitResolutionStatus.RESOLVED:
        return InputAdministrativeLevel.AUTONOMOUS_COMMUNITY.value
    return None


def resolve_project_locations(project_locations: pd.DataFrame) -> pd.DataFrame:
    contexts, preliminary = build_publication_contexts(project_locations)
    records: list[dict[str, Any]] = []

    for idx, row in project_locations.iterrows():
        pre = preliminary.loc[idx]
        level: InputAdministrativeLevel = pre['input_level']
        context = contexts[publication_context_key(row)]
        explicit_province: ProvinceLookupResult = pre['province_hint_lookup']
        explicit_ac: AutonomousCommunityLookupResult = pre['ac_hint_lookup']

        province_hint_source = 'explicit' if explicit_province.resolution_status == AdministrativeUnitResolutionStatus.RESOLVED else 'none'
        ac_hint_source = 'explicit' if explicit_ac.resolution_status == AdministrativeUnitResolutionStatus.RESOLVED else 'none'

        effective_province = explicit_province
        if (
            level != InputAdministrativeLevel.AUTONOMOUS_COMMUNITY
            and explicit_province.resolution_status
            in {
                AdministrativeUnitResolutionStatus.NOT_PROVIDED,
                AdministrativeUnitResolutionStatus.NOT_FOUND,
            }
            and context.province is not None
        ):
            effective_province = inherited_province_result(
                context.province,
                sources=list(context.province_sources),
            )
            province_hint_source = "publication_context"

        effective_ac = explicit_ac
        if explicit_ac.resolution_status in {AdministrativeUnitResolutionStatus.NOT_PROVIDED, AdministrativeUnitResolutionStatus.NOT_FOUND} and context.autonomous_community is not None:
            effective_ac = inherited_ac_result(context.autonomous_community, sources=list(context.autonomous_community_sources))
            ac_hint_source = 'publication_context'

        base = row.to_dict()
        base.update(CANONICAL_LOCATION_COLUMNS)
        base["location_resolver_version"] = LOCATION_RESOLVER_VERSION
        base["municipality_dimension_source"] = MUNICIPALITY_DIMENSION_SOURCE
        base["province_hint_lookup_status"] = (
            explicit_province.resolution_status.value
        )
        base["province_hint_lookup_matched_by"] = explicit_province.matched_by
        base["province_hint_lookup_reason"] = explicit_province.reason
        base["province_hint_lookup_candidate_codes"] = [
            candidate.ine_province_code
            for candidate in explicit_province.candidates
        ]
        base["autonomous_community_hint_lookup_status"] = (
            explicit_ac.resolution_status.value
        )
        base["autonomous_community_hint_lookup_matched_by"] = (
            explicit_ac.matched_by
        )
        base["autonomous_community_hint_lookup_reason"] = explicit_ac.reason
        base["autonomous_community_hint_lookup_candidate_codes"] = [
            candidate.ine_autonomous_community_code
            for candidate in explicit_ac.candidates
        ]
        base['input_administrative_level'] = level.value
        base['input_administrative_level_matched_by'] = pre['input_level_matched_by']
        base['input_administrative_level_reason'] = pre['input_level_reason']
        base['publication_context_status'] = context.status
        base['publication_context_reason'] = context.reason
        base['publication_context_province_candidate_codes'] = list(context.province_candidate_codes)
        base['publication_context_autonomous_community_candidate_codes'] = list(context.autonomous_community_candidate_codes)
        base['province_hint_source'] = province_hint_source
        base['autonomous_community_hint_source'] = ac_hint_source

        if level == InputAdministrativeLevel.PROVINCE:
            municipality_result = not_provided_municipality(row.get('municipality_raw'), level)
            raw_province: ProvinceLookupResult = pre['raw_province_lookup']
            if raw_province.resolution_status == AdministrativeUnitResolutionStatus.RESOLVED:
                province_result = reconcile_province(effective_province, raw_province.resolved, source='reclassified_raw_location')
            else:
                province_result = effective_province
            if province_result.resolution_status == AdministrativeUnitResolutionStatus.RESOLVED:
                base.update(province_result.resolved.model_dump())
                ac_result = reconcile_ac(effective_ac, canonical_ac_from_province(province_result.resolved), source='province')
            else:
                ac_result = effective_ac
                if ac_result.resolution_status == AdministrativeUnitResolutionStatus.RESOLVED:
                    base.update(ac_result.resolved.model_dump())

        elif level == InputAdministrativeLevel.AUTONOMOUS_COMMUNITY:
            municipality_result = not_provided_municipality(row.get('municipality_raw'), level)
            province_result = not_provided_province(row.get('province_hint_raw'), level)
            raw_ac: AutonomousCommunityLookupResult = pre['raw_ac_lookup']
            if raw_ac.resolution_status == AdministrativeUnitResolutionStatus.RESOLVED:
                ac_result = reconcile_ac(effective_ac, raw_ac.resolved, source='reclassified_raw_location')
            else:
                ac_result = effective_ac
            if ac_result.resolution_status == AdministrativeUnitResolutionStatus.RESOLVED:
                base.update(ac_result.resolved.model_dump())

        else:
            province_code = effective_province.resolved.ine_province_code if effective_province.resolution_status == AdministrativeUnitResolutionStatus.RESOLVED else None
            ac_code = effective_ac.resolved.ine_autonomous_community_code if effective_ac.resolution_status == AdministrativeUnitResolutionStatus.RESOLVED else None
            if province_code is not None and ac_code is not None and effective_province.resolved.ine_autonomous_community_code != ac_code:
                ac_code = None
            municipality_result = resolve_municipality(
                row.get('municipality_raw'), row.get('province_hint_raw'), row.get('autonomous_community_hint_raw'),
                province_code=province_code, ac_code=ac_code,
            )
            province_result = effective_province
            ac_result = effective_ac
            if municipality_result.resolution_status == AdministrativeUnitResolutionStatus.RESOLVED:
                base.update(municipality_result.resolved.model_dump())
                province_result = reconcile_province(effective_province, canonical_province_from_municipality(municipality_result.resolved), source='municipality')
                ac_result = reconcile_ac(effective_ac, canonical_ac_from_province(province_result.resolved), source='municipality')
            elif province_result.resolution_status == AdministrativeUnitResolutionStatus.RESOLVED:
                base.update(province_result.resolved.model_dump())
                ac_result = reconcile_ac(effective_ac, canonical_ac_from_province(province_result.resolved), source='province')
            elif ac_result.resolution_status == AdministrativeUnitResolutionStatus.RESOLVED:
                base.update(ac_result.resolved.model_dump())

        add_metadata(base, 'municipality', municipality_result)
        add_metadata(base, 'province', province_result)
        add_metadata(base, 'autonomous_community', ac_result)
        status = aggregate_location_status(
            municipality_result,
            province_result,
            ac_result,
        )
        base['location_resolution_status'] = status.value
        base['location_resolution_level'] = resolved_level(municipality_result, province_result, ac_result)
        base['location_resolution_matched_by'] = 'hierarchical_reconciliation_with_publication_context'
        base['location_resolution_reason'] = build_location_resolution_reason(
            status,
            municipality_result,
            province_result,
            ac_result,
        )
        records.append(base)

    return pd.DataFrame(records)


## 9. Validaciones estructurales

In [ ]:
def validate_resolved_locations(df: pd.DataFrame) -> None:
    """Valida integridad territorial y coherencia de los estados agregados."""
    required = {
        "project_location_id",
        "input_administrative_level",
        "location_resolution_status",
        "municipality_resolution_status",
        "province_resolution_status",
        "autonomous_community_resolution_status",
        "ine_municipality_code",
        "ine_province_code",
        "ine_autonomous_community_code",
    }
    validate_required_columns(df, required)

    if df["project_location_id"].duplicated().any():
        raise ValueError("project_location_id no es único.")

    valid_unit_statuses = {
        status.value for status in AdministrativeUnitResolutionStatus
    }
    valid_location_statuses = {
        status.value for status in LocationResolutionStatus
    }

    province_to_ac = dict(
        provinces[["cpro", "cauto"]].itertuples(index=False, name=None)
    )
    errors: list[str] = []

    for idx, row in df.iterrows():
        municipality_code = row.get("ine_municipality_code")
        province_code = row.get("ine_province_code")
        ac_code = row.get("ine_autonomous_community_code")

        unit_status_values = (
            row["municipality_resolution_status"],
            row["province_resolution_status"],
            row["autonomous_community_resolution_status"],
        )

        invalid_unit_statuses = set(unit_status_values) - valid_unit_statuses
        if invalid_unit_statuses:
            errors.append(
                f"fila {idx}: estados administrativos no válidos "
                f"{sorted(invalid_unit_statuses)}"
            )
            continue

        actual_location_status = row["location_resolution_status"]
        if actual_location_status not in valid_location_statuses:
            errors.append(
                f"fila {idx}: location_resolution_status no válido "
                f"{actual_location_status!r}"
            )
            continue

        expected_location_status = aggregate_unit_resolution_statuses(
            tuple(
                AdministrativeUnitResolutionStatus(value)
                for value in unit_status_values
            )
        ).value
        if actual_location_status != expected_location_status:
            errors.append(
                f"fila {idx}: location_resolution_status="
                f"{actual_location_status!r}, esperado="
                f"{expected_location_status!r} para {unit_status_values}"
            )

        if not is_null_like(municipality_code):
            if (
                is_null_like(province_code)
                or not str(municipality_code).startswith(
                    str(province_code).zfill(2)
                )
            ):
                errors.append(
                    f"fila {idx}: municipio {municipality_code} no "
                    f"pertenece a provincia {province_code}"
                )

        if not is_null_like(province_code):
            expected_ac = province_to_ac.get(str(province_code).zfill(2))
            if (
                expected_ac is None
                or is_null_like(ac_code)
                or str(ac_code).zfill(2) != expected_ac
            ):
                errors.append(
                    f"fila {idx}: provincia {province_code} no "
                    f"pertenece a comunidad {ac_code}"
                )

        if (
            row["municipality_resolution_status"] == "resolved"
            and (
                is_null_like(municipality_code)
                or is_null_like(province_code)
                or is_null_like(ac_code)
            )
        ):
            errors.append(
                f"fila {idx}: municipio resuelto sin jerarquía completa"
            )

        if (
            row["province_resolution_status"] == "resolved"
            and (is_null_like(province_code) or is_null_like(ac_code))
        ):
            errors.append(
                f"fila {idx}: provincia resuelta sin comunidad"
            )

        if (
            row["autonomous_community_resolution_status"] == "resolved"
            and is_null_like(ac_code)
        ):
            errors.append(
                f"fila {idx}: comunidad autónoma resuelta sin código"
            )

    if errors:
        raise ValueError(
            "Localizaciones inconsistentes: " + "; ".join(errors[:20])
        )


In [ ]:
project_locations_resolved = resolve_project_locations(
    project_locations
)

validate_resolved_locations(project_locations_resolved)

print(
    f"Localizaciones resueltas: {len(project_locations_resolved):,}"
)


## 10. Pruebas de regresión, aislamiento por publicación y determinismo


In [ ]:
def build_synthetic_location(
    *,
    project_location_id: str,
    project_mention_id: str,
    municipality_raw: str,
    province_hint_raw: str | None = None,
    autonomous_community_hint_raw: str | None = None,
    location_evidence: str,
    location_index: int = 1,
) -> dict[str, Any]:
    return {
        "project_location_id": project_location_id,
        "project_mention_id": project_mention_id,
        "event_id": f"{project_mention_id}_event",
        "identificador_boe": "SYNTHETIC",
        "fecha_publicacion": "2026-07-10",
        "location_index": location_index,
        "municipality_raw": municipality_raw,
        "municipality_raw_norm": normalize_text_or_none(municipality_raw),
        "province_hint_raw": province_hint_raw,
        "province_hint_raw_norm": normalize_text_or_none(
            province_hint_raw
        ),
        "autonomous_community_hint_raw": (
            autonomous_community_hint_raw
        ),
        "autonomous_community_hint_raw_norm": normalize_text_or_none(
            autonomous_community_hint_raw
        ),
        "location_evidence": location_evidence,
    }


# Casos reales del conjunto adjunto.
as_pontes = project_locations_resolved.loc[
    project_locations_resolved["municipality_raw_norm"].str.contains(
        "pontes",
        na=False,
    )
]
if not as_pontes.empty:
    assert as_pontes["ine_municipality_code"].eq("15070").all()

cerdeira = project_locations_resolved.loc[
    project_locations_resolved["municipality_raw_norm"].eq("cerdeira")
]
if not cerdeira.empty:
    assert cerdeira["ine_municipality_code"].eq("15022").all()
    assert cerdeira["municipality_resolution_matched_by"].str.startswith(
        "municipality_fuzzy"
    ).all()

tharsis = project_locations_resolved.loc[
    project_locations_resolved["municipality_raw_norm"].eq("tharsis")
]
if not tharsis.empty:
    assert tharsis["municipality_resolution_status"].eq("not_found").all()
    assert tharsis["ine_province_code"].eq("21").all()
    assert tharsis["ine_autonomous_community_code"].eq("01").all()
    assert tharsis["location_resolution_status"].eq(
        "partially_resolved"
    ).all()

province_huelva_rows = project_locations_resolved.loc[
    project_locations_resolved["municipality_raw_norm"].eq("huelva")
    & project_locations_resolved["location_evidence"]
    .map(normalize_text_or_none)
    .str.contains("provincia de huelva", na=False)
]
if not province_huelva_rows.empty:
    assert province_huelva_rows["input_administrative_level"].eq(
        "province"
    ).all()
    assert province_huelva_rows["ine_municipality_code"].isna().all()
    assert province_huelva_rows["ine_province_code"].eq("21").all()
    assert province_huelva_rows["location_resolution_status"].eq(
        "partially_resolved"
    ).all()

andalucia_rows = project_locations_resolved.loc[
    project_locations_resolved["municipality_raw_norm"].eq("andalucia")
]
if not andalucia_rows.empty:
    assert andalucia_rows["input_administrative_level"].eq(
        "autonomous_community"
    ).all()
    assert andalucia_rows["ine_municipality_code"].isna().all()
    assert andalucia_rows["ine_province_code"].isna().all()
    assert andalucia_rows["ine_autonomous_community_code"].eq("01").all()
    assert andalucia_rows["location_resolution_status"].eq(
        "partially_resolved"
    ).all()


# Un municipio homónimo de su provincia debe mantenerse como municipio cuando
# la evidencia lo etiqueta explícitamente como tal.
true_huelva = pd.DataFrame(
    [
        build_synthetic_location(
            project_location_id="synthetic_huelva_municipality",
            project_mention_id="synthetic_huelva_project",
            municipality_raw="Huelva",
            province_hint_raw="Huelva",
            location_evidence=(
                "municipio de Huelva, provincia de Huelva"
            ),
        )
    ]
)
true_huelva_resolved = resolve_project_locations(true_huelva)
assert true_huelva_resolved.loc[0, "input_administrative_level"] == (
    "municipality"
)
assert true_huelva_resolved.loc[0, "ine_municipality_code"] == "21041"


# Una pista provincial contradictoria debe producir conflict y conservar la
# jerarquía del municipio, que es la unidad más específica.
conflict_case = pd.DataFrame(
    [
        build_synthetic_location(
            project_location_id="synthetic_conflict",
            project_mention_id="synthetic_conflict_project",
            municipality_raw="Cedeira",
            province_hint_raw="Huelva",
            location_evidence=(
                "municipio de Cedeira, provincia de Huelva"
            ),
        )
    ]
)
conflict_resolved = resolve_project_locations(conflict_case)
assert conflict_resolved.loc[0, "ine_municipality_code"] == "15022"
assert conflict_resolved.loc[0, "province_resolution_status"] == "conflict"
assert conflict_resolved.loc[0, "location_resolution_status"] == "conflict"


# En un proyecto con municipios de varias provincias no se debe propagar una
# provincia común a una localidad no resuelta.
multi_province_case = pd.DataFrame(
    [
        build_synthetic_location(
            project_location_id="synthetic_multi_1",
            project_mention_id="synthetic_multi_project",
            municipality_raw="Cedeira",
            location_evidence="Cedeira",
            location_index=1,
        ),
        build_synthetic_location(
            project_location_id="synthetic_multi_2",
            project_mention_id="synthetic_multi_project",
            municipality_raw="Alosno",
            location_evidence="Alosno",
            location_index=2,
        ),
        build_synthetic_location(
            project_location_id="synthetic_multi_3",
            project_mention_id="synthetic_multi_project",
            municipality_raw="Tharsis",
            location_evidence="Tharsis",
            location_index=3,
        ),
    ]
)
multi_resolved = resolve_project_locations(
    multi_province_case
)
assert multi_resolved["publication_context_status"].eq("ambiguous").all()
multi_tharsis = multi_resolved.loc[
    multi_resolved["municipality_raw_norm"].eq("tharsis")
].iloc[0]
assert is_null_like(multi_tharsis["ine_province_code"])
assert multi_tharsis["location_resolution_status"] == "not_found"



# El contexto no puede propagarse entre publicaciones diferentes aunque se
# reutilicen accidentalmente los mismos identificadores internos.
publication_isolation_case = pd.DataFrame(
    [
        {
            **build_synthetic_location(
                project_location_id="synthetic_publication_1",
                project_mention_id="shared_project_mention",
                municipality_raw="Cedeira",
                location_evidence="Cedeira",
            ),
            "identificador_boe": "SYNTHETIC-A",
            "event_id": "shared_event",
        },
        {
            **build_synthetic_location(
                project_location_id="synthetic_publication_2",
                project_mention_id="shared_project_mention",
                municipality_raw="Tharsis",
                location_evidence="Tharsis",
            ),
            "identificador_boe": "SYNTHETIC-B",
            "event_id": "shared_event",
        },
    ]
)
publication_isolation_resolved = resolve_project_locations(
    publication_isolation_case
)
isolated_tharsis = publication_isolation_resolved.loc[
    publication_isolation_resolved["project_location_id"].eq(
        "synthetic_publication_2"
    )
].iloc[0]
assert is_null_like(isolated_tharsis["ine_province_code"])
assert isolated_tharsis["location_resolution_status"] == "not_found"


# Contrato global: `resolved` exige exactamente tres niveles resueltos y
# `partially_resolved` exige uno o dos, salvo conflicto o ambigüedad.
UNIT_STATUS_COLUMNS = [
    "municipality_resolution_status",
    "province_resolution_status",
    "autonomous_community_resolution_status",
]
resolved_count = project_locations_resolved[UNIT_STATUS_COLUMNS].eq(
    AdministrativeUnitResolutionStatus.RESOLVED.value
).sum(axis=1)

assert project_locations_resolved.loc[
    project_locations_resolved["location_resolution_status"].eq("resolved"),
    UNIT_STATUS_COLUMNS,
].eq("resolved").all(axis=None)

partial_mask = project_locations_resolved[
    "location_resolution_status"
].eq("partially_resolved")
assert resolved_count.loc[partial_mask].isin({1, 2}).all()

assert not (
    project_locations_resolved["location_resolution_status"].eq("resolved")
    & resolved_count.ne(3)
).any()


# El resultado no depende del orden de las filas.
shuffled_resolved = resolve_project_locations(
    project_locations.sample(frac=1, random_state=42).reset_index(drop=True)
)

DETERMINISM_COLUMNS = [
    "project_location_id",
    "input_administrative_level",
    "municipality",
    "ine_municipality_code",
    "municipality_resolution_status",
    "province",
    "ine_province_code",
    "province_resolution_status",
    "autonomous_community",
    "ine_autonomous_community_code",
    "autonomous_community_resolution_status",
    "location_resolution_status",
    "location_resolution_level",
]

expected = (
    project_locations_resolved[DETERMINISM_COLUMNS]
    .sort_values("project_location_id")
    .reset_index(drop=True)
)
actual = (
    shuffled_resolved[DETERMINISM_COLUMNS]
    .sort_values("project_location_id")
    .reset_index(drop=True)
)
pd.testing.assert_frame_equal(expected, actual)

print("Pruebas de regresión y determinismo superadas.")


## 11. Guardar resultados

Solo se persiste `project_locations_resolved.parquet`. El contexto administrativo utilizado durante la resolución permanece como estado interno y queda documentado en las columnas `publication_context_*` de cada fila.


In [ ]:
save_parquet(
    project_locations_resolved,
    PROJECT_LOCATIONS_RESOLVED_PATH,
)

print(PROJECT_LOCATIONS_RESOLVED_PATH.relative_to(PROJECT_ROOT))


## 12. Resumen de resultados

In [ ]:
for status_column in [
    "input_administrative_level",
    "location_resolution_status",
    "municipality_resolution_status",
    "province_resolution_status",
    "autonomous_community_resolution_status",
]:
    display(
        project_locations_resolved[status_column]
        .value_counts(dropna=False)
        .rename_axis(status_column)
        .reset_index(name="n")
    )


In [ ]:
audit_view = project_locations_resolved.copy()
audit_view["resolved_administrative_levels"] = audit_view[
    [
        "municipality_resolution_status",
        "province_resolution_status",
        "autonomous_community_resolution_status",
    ]
].eq("resolved").sum(axis=1)

AUDIT_COLUMNS = [
    "identificador_boe",
    "project_location_id",
    "municipality_raw",
    "province_hint_raw",
    "autonomous_community_hint_raw",
    "input_administrative_level",
    "municipality",
    "ine_municipality_code",
    "municipality_resolution_status",
    "municipality_resolution_matched_by",
    "province",
    "ine_province_code",
    "province_resolution_status",
    "province_hint_source",
    "autonomous_community",
    "ine_autonomous_community_code",
    "autonomous_community_resolution_status",
    "location_resolution_status",
    "resolved_administrative_levels",
    "location_resolution_level",
]

focus_mask = audit_view["municipality_raw_norm"].str.contains(
    "pontes|cerdeira|tharsis|huelva|andalucia",
    na=False,
)

display(
    audit_view.loc[
        focus_mask,
        AUDIT_COLUMNS,
    ]
)


In [ ]:
display(
    audit_view.loc[
        audit_view["location_resolution_status"]
        != LocationResolutionStatus.RESOLVED.value,
        AUDIT_COLUMNS,
    ]
)


## 13. Interpretación de los estados

### Estados por unidad administrativa

Cada columna `*_resolution_status` describe exclusivamente el resultado del nivel correspondiente:

- `resolved`: la unidad se identificó de forma unívoca;
- `ambiguous`: existen varios candidatos compatibles;
- `not_found`: se intentó resolver la unidad, pero no se encontró un candidato suficiente;
- `not_provided`: la unidad no estaba aportada o la mención fue reclasificada a otro nivel;
- `conflict`: una pista explícita contradice la jerarquía canónica determinada por una unidad más específica.

### Estado global de la localización

`location_resolution_status` mide la **completitud y coherencia de los tres niveles administrativos**:

- `resolved`: municipio, provincia y comunidad autónoma están `resolved`;
- `partially_resolved`: uno o dos niveles están `resolved` y no existe conflicto ni ambigüedad;
- `ambiguous`: al menos un nivel es ambiguo y no existe conflicto;
- `conflict`: existe al menos un conflicto jerárquico;
- `not_found`: ningún nivel está resuelto y al menos uno no pudo encontrarse;
- `not_provided`: ningún nivel está resuelto y no hubo una búsqueda fallida.

`location_resolution_level` indica el **nivel más específico resuelto**, pero no determina por sí solo el estado global. Por ejemplo, una mención de `Andalucía` puede tener `location_resolution_level = autonomous_community` y `location_resolution_status = partially_resolved`.

Las columnas `publication_context_*` permiten auditar el contexto temporal utilizado dentro de la misma publicación, evento y mención de proyecto. Este contexto no se persiste como tabla independiente.

| Niveles resueltos | Situación adicional     | Estado global        |
| ----------------: | ----------------------- | -------------------- |
|                 3 | —                       | `resolved`           |
|             1 o 2 | —                       | `partially_resolved` |
|                 0 | algún nivel `not_found` | `not_found`          |
|                 0 | todos `not_provided`    | `not_provided`       |
|        cualquiera | algún `ambiguous`       | `ambiguous`          |
|        cualquiera | algún `conflict`        | `conflict`           |
